In [3]:
from google.colab import drive
drive.mount('/content/drive')

# ========================
# PARTE 0: INSTALAÇÕES (Google Colab)
# ========================
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium > /dev/null
!pip install pandas openpyxl requests > /dev/null

# ========================
# PARTE 1: IMPORTS E CONFIGURAÇÕES
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
import os, time, requests, zipfile
from datetime import datetime

options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# ========================
# PARTE 2: CONFIGURAÇÃO ALGOLIA
# ========================
ALGOLIA_APP_ID = "1CF3ZT43ZU"
ALGOLIA_API_KEY = "150c68d1c61fc1835826a57a203dab72"
ALGOLIA_INDEX = "production_products"

headers_algolia = {
    "Content-Type": "application/json",
    "X-Algolia-Application-Id": ALGOLIA_APP_ID,
    "X-Algolia-API-Key": ALGOLIA_API_KEY,
}

termos_busca = [
    "porcelanato",
    "pisos ceramicos",
    "pisos vinilicos",
    "pisos laminados",
    "piso emborrachado",
    "ladrilho hidraulico",
    "ladrilhos para calçada",
    "pisogramas",
    "pisos intertravado",
    "carpetes",
    "pedras para piso"
]

def buscar_links_por_categoria(termos, max_paginas=50, limite_por_termo=10):
    links = set()
    for termo in termos:
        print(f"🔍 Buscando produtos para: {termo}")
        contador = 0
        for pagina in range(max_paginas):
            url = f"https://{ALGOLIA_APP_ID}-dsn.algolia.net/1/indexes/{ALGOLIA_INDEX}/query"
            payload = {
                "params": f"query={termo}&page={pagina}&hitsPerPage=100"
            }
            res = requests.post(url, headers=headers_algolia, json=payload)
            if res.status_code != 200:
                print(f"⚠️ Erro na requisição para '{termo}' página {pagina}")
                break
            data = res.json()
            hits = data.get("hits", [])
            if not hits:
                break
            for item in hits:
                relative_url = item.get("url")
                if relative_url:
                    full_url = f"https://www.leroymerlin.com.br{relative_url}" if not relative_url.startswith("http") else relative_url
                    if full_url not in links:
                        links.add(full_url)
                        contador += 1
                        if contador >= limite_por_termo:
                            break
            if contador >= limite_por_termo or pagina >= data.get("nbPages", 0) - 1:
                break
    return links

print("\n🔍 Coletando TODOS os links via API Algolia por categoria...")
todos_links = buscar_links_por_categoria(termos_busca)
print(f"✅ Total de links coletados: {len(todos_links)}\n")

# ========================
# PARTE 3: EXTRAÇÃO DE DADOS DOS PRODUTOS
# ========================
def extrair_dados_produto(url):
    driver.get(url)
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        return {'URL': url, 'Erro': 'Timeout'}

    dados = {'URL': url}
    dados['Nome'] = driver.find_element(By.CSS_SELECTOR, "h1").text if driver.find_elements(By.CSS_SELECTOR, "h1") else None
    dados['Código'] = url.split("_")[-1] if "_" in url else None

    preco_txt = None
    precos_candidatos = [
        '[data-cy="product-price"]',
        '[data-cy="product-price-integer"]',
        '[data-cy="product-price-sale"]',
        '.price__sales',
        '.value__integer',
        '.price',
        '.price__value'
    ]
    for sel in precos_candidatos:
        elems = driver.find_elements(By.CSS_SELECTOR, sel)
        if elems:
            text = elems[0].text.strip()
            if text and text.lower() != "grátis":
                preco_txt = text
                break

    if preco_txt:
        clean = preco_txt.replace("R$", "").replace("/m²", "").replace(" ", "").replace(".", "").replace(",", ".").strip()
        try:
            preco_val = float(clean)
            dados['Preço Promocional'] = f"{preco_val:.2f}".replace(".", ",")
        except:
            dados['Preço Promocional'] = "N/A"
    else:
        dados['Preço Promocional'] = "N/A"

    dados['Descrição'] = driver.find_element(By.CSS_SELECTOR, 'meta[name="description"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[name="description"]') else None
    dados['Imagem'] = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:image"]') else "N/A"

    specs = {}
    for tr in driver.find_elements(By.CSS_SELECTOR, 'table tr'):
        try:
            k = tr.find_element(By.CSS_SELECTOR, 'th').text.strip()
            v = tr.find_element(By.CSS_SELECTOR, 'td').text.strip()
            specs[k] = v
        except:
            continue
    dados['Especificações'] = "\n".join(f"{k}: {v}" for k, v in specs.items())
    dados['Loja'] = "Leroy Merlim"
    return dados

# ========================
# PARTE 4: COLETA DOS PRODUTOS
# ========================
todos = []
for i, link in enumerate(sorted(todos_links), 1):
    print(f"🔍 {i}/{len(todos_links)} -> {link}")
    try:
        dados = extrair_dados_produto(link)
        todos.append(dados)
    except Exception as err:
        print("Erro:", err)
driver.quit()

# ========================
# PARTE 5: SALVAR PLANILHA E CSV COM ORDEM DEFINIDA
# ========================
data_nome = datetime.now().strftime("%Y-%m-%d")
pasta_base_local = "leroymerelin_coleta"
os.makedirs(pasta_base_local, exist_ok=True)

arquivo_excel = f"{pasta_base_local}/leroymerelin_{data_nome}.xlsx"
arquivo_csv = f"{pasta_base_local}/leroymerelin_{data_nome}.csv"

colunas_ordenadas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Especificações', 'Imagem', 'URL', 'Loja']
df = pd.DataFrame(todos)
df.drop(columns=[c for c in df.columns if c in ['Erro']], inplace=True, errors='ignore')
df = df[colunas_ordenadas]
df.to_excel(arquivo_excel, index=False)
df.to_csv(arquivo_csv, index=False, sep=";")

print(f"📄 Excel salvo em: {arquivo_excel}")
print(f"📄 CSV salvo em: {arquivo_csv}")

# ========================
# PARTE 6: BAIXAR IMAGENS NO GOOGLE DRIVE E ZIPAR
# ========================
# Caminho no Google Drive
base_drive = "/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY"
os.makedirs(base_drive, exist_ok=True)

pasta_imagens = os.path.join(base_drive, f"imagens_leroymerelin_{data_nome}")
os.makedirs(pasta_imagens, exist_ok=True)

for _, r in df.iterrows():
    url_img = r['Imagem']
    cod = r['Código']
    nome_arquivo = f"{cod}.png"
    caminho_arquivo = os.path.join(pasta_imagens, nome_arquivo)

    if url_img and url_img != "N/A" and not os.path.exists(caminho_arquivo):
        for tentativa in range(3):
            try:
                resp = requests.get(url_img, headers={"User-Agent": "Mozilla/5.0"}, timeout=10, stream=True)
                if resp.status_code == 200:
                    with open(caminho_arquivo, "wb") as f:
                        for cb in resp.iter_content(1024):
                            f.write(cb)
                    print(f"✅ Imagem baixada: {nome_arquivo}")
                    break
            except Exception as e:
                print(f"⚠️ Tentativa {tentativa + 1} falhou para {cod}: {e}")
                time.sleep(2)
    else:
        print(f"⏩ Imagem já existe: {nome_arquivo}")

# Compactar imagens em .zip no mesmo local
zipname = os.path.join(base_drive, f"imagens_leroymerelin_{data_nome}.zip")
with zipfile.ZipFile(zipname, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(pasta_imagens):
        z.write(os.path.join(pasta_imagens, fn), fn)

print(f"📦 Imagens zipadas em: {zipname}")


Mounted at /content/drive
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file

🔍 Coletando TODOS os links via API Algolia por categoria...
🔍 Buscando produtos para: porcelanato
🔍 Buscando produtos para: pisos ceramicos
🔍 Buscando produtos para: pisos vinilicos
🔍 Buscando produtos para: pisos laminados
🔍 Buscando produtos para: piso emborrachado
🔍 Buscando produtos para: ladrilho hidraulico
🔍 Buscando produtos para: ladrilhos para calçada
🔍 Buscando produtos para: pisogramas
🔍 Buscando produtos para: pisos intertravado
🔍 Buscando produtos para: carpetes
🔍 Buscando produtos para: pedras para piso
✅ Total de links coletados: 110

🔍 1/110 -> https://www.leroymerlin.com.br/carpete-autolour-bege-com-resina--por-metro-quadrado-_1571820494
🔍 2/110 -> https://www.leroy